In [ ]:
import pandas as pd
import os
import glob
#import mahotas
import cv2
from pathlib import Path
import numpy as np
from tqdm import tqdm
from PIL import Image
import requests

from sklearn.model_selection import StratifiedGroupKFold

from tensorflow.keras.applications import ResNet50, VGG19, ConvNeXtBase, DenseNet121, DenseNet169
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.applications.vgg19 import preprocess_input
from tensorflow.keras.applications.convnext import preprocess_input
from tensorflow.keras.applications.densenet import preprocess_input

from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tensorflow.keras.utils import load_img, img_to_array

from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from typing import List, Union, Literal
import skimage

from scipy.signal import convolve2d
from scipy.spatial.distance import cdist

In [ ]:
from transformers import ViTFeatureExtractor, ViTModel, AutoImageProcessor, AutoModel, AutoFeatureExtractor, ViTMSNModel, SwinModel
import torch
import gc

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
benignant_images_path = "./data/benignant_images"
malignant_images_path = "./data/malignant_images"

In [ ]:
benignant_images = []
malignant_images = []
for file in glob.glob(os.path.join(benignant_images_path, "*.jpg")):
    benignant_images.append(file)

for file in glob.glob(os.path.join(malignant_images_path, "*.jpg")):
    malignant_images.append(file)

In [ ]:
def feature_extract_ViT_Base(images_path, save_filename, out_dir=None, verbose=False):

    if out_dir is None:
        out_dir = os.path.join(os.getcwd(), "features")
    os.makedirs(out_dir, exist_ok=True)

    feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224-in21k')
    model = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
    model.eval()

    model.to(device)

    filenames = []
    features_list = []

    for i, img_path in enumerate(tqdm(images_path, desc="Extracting ViT Base Features")):
        if isinstance(img_path, (str, Path)):
            filename = Path(img_path).name
        else:
            filename = getattr(img_path, "filename", None) or getattr(img_path, "name", None) or f"image_{i}"
        try:
            image = Image.open(img_path)

            inputs = feature_extractor(images=image, return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = model(**inputs)


            pooled_features = outputs.pooler_output

            # Move de volta para a CPU, converte para NumPy e "achata"
            features_np = pooled_features.cpu().detach().numpy().flatten()


            filenames.append(filename)
            features_list.append(features_np)

            if verbose and (i + 1) % 50 == 0:
                print(f"Processed {i+1} images")

        except Exception as e:
            print(f"Warning: failed to process {img_path}: {e}")
            continue

    if len(features_list) == 0:
        raise ValueError("No features extracted. Check your images_path list and image readability.")

    # Empilha todas as features em matriz (n_samples x 4096)
    features_array = np.vstack(features_list)
    features_dim = features_array.shape[1]

    # Cria DataFrame de features
    feature_cols = [f"f{i}" for i in range(features_dim)]
    df_features = pd.DataFrame(features_array, columns=feature_cols)
    df_features.insert(0, "filename", filenames)

    # Salva CSV
    out_path = os.path.join(out_dir, f"vitbase_{save_filename}.csv")
    df_features.to_csv(out_path, index=False)

    if verbose:
        print(f"Saved features to {out_path} (shape={df_features.shape})")

    return df_features

In [ ]:
out_dir = "/content/drive/MyDrive/TCC/features"

In [ ]:
benignant_vitbase = feature_extract_ViT_Base(benignant_images, "benignant_features", out_dir)

In [ ]:
malignant_vitbase = feature_extract_ViT_Base(malignant_images, "malignant_features", out_dir)